## Libraries

In [1]:
import pandas as pd
from pathlib import Path

## Load the CSV File

In [2]:
try:
    df = pd.read_csv(r'C:\Users\Alondra Soto\OneDrive\Documents\Development\Sample_Sales_Data\database\original_csv\sales_data_sample.csv', encoding='cp1252')
except FileNotFoundError:
    print("The specified file was not found.")
else:
    print(df.head())
    print(df.info())


   ORDERNUMBER  QUANTITYORDERED  PRICEEACH  ORDERLINENUMBER    SALES  \
0        10107               30      95.70                2  2871.00   
1        10121               34      81.35                5  2765.90   
2        10134               41      94.74                2  3884.34   
3        10145               45      83.26                6  3746.70   
4        10159               49     100.00               14  5205.27   

         ORDERDATE   STATUS  QTR_ID  MONTH_ID  YEAR_ID  ...  \
0   2/24/2003 0:00  Shipped       1         2     2003  ...   
1    5/7/2003 0:00  Shipped       2         5     2003  ...   
2    7/1/2003 0:00  Shipped       3         7     2003  ...   
3   8/25/2003 0:00  Shipped       3         8     2003  ...   
4  10/10/2003 0:00  Shipped       4        10     2003  ...   

                    ADDRESSLINE1  ADDRESSLINE2           CITY STATE  \
0        897 Long Airport Avenue           NaN            NYC    NY   
1             59 rue de l'Abbaye           NaN

In [3]:
# View available columns
print("Available columns:")
print(df.columns.tolist())
print("\nColumn count:", len(df.columns))
print("\nDataset shape:", df.shape)

Available columns:
['ORDERNUMBER', 'QUANTITYORDERED', 'PRICEEACH', 'ORDERLINENUMBER', 'SALES', 'ORDERDATE', 'STATUS', 'QTR_ID', 'MONTH_ID', 'YEAR_ID', 'PRODUCTLINE', 'MSRP', 'PRODUCTCODE', 'CUSTOMERNAME', 'PHONE', 'ADDRESSLINE1', 'ADDRESSLINE2', 'CITY', 'STATE', 'POSTALCODE', 'COUNTRY', 'TERRITORY', 'CONTACTLASTNAME', 'CONTACTFIRSTNAME', 'DEALSIZE']

Column count: 25

Dataset shape: (2823, 25)


## Customer And Address

In [5]:
# Show the preprocessing used for the Customer section
print("1) customer_base")
# Preprocessing for Customer and Address

# Define output directory
output_dir = Path(r'C:\Users\Alondra Soto\OneDrive\Documents\Development\Sample_Sales_Data\database\preprocessing_tables')
output_dir.mkdir(parents=True, exist_ok=True)

# Extract customer and address columns
customer_columns = ['CUSTOMERNAME', 'PHONE', 'ADDRESSLINE1', 'ADDRESSLINE2', 'CITY', 'STATE', 'POSTALCODE', 'COUNTRY', 'TERRITORY']
customer_base = df[customer_columns].drop_duplicates().reset_index(drop=True)

# Separate address columns
address_columns = ['ADDRESSLINE1', 'ADDRESSLINE2', 'CITY', 'STATE', 'POSTALCODE', 'COUNTRY', 'TERRITORY']
address_unique = customer_base[address_columns].drop_duplicates().reset_index(drop=True)
address_unique.insert(0, 'ADDRESS_ID', range(1, len(address_unique) + 1))

# Map address IDs back to customer base
customer_address_map = customer_base.merge(address_unique, on=address_columns, how='left')

# Create address table
address_table = address_unique.copy()

# Create customer table
customer_table = customer_address_map[['CUSTOMERNAME', 'PHONE', 'ADDRESS_ID']].drop_duplicates().reset_index(drop=True)
customer_table.insert(0, 'CUSTOMER_ID', range(1, len(customer_table) + 1))

print(customer_base.head())
print("Shape:", customer_base.shape)

print("\n2) customer_address_map")
print(customer_address_map.head())
print("Shape:", customer_address_map.shape)

print("\n3) address_table")
print(address_table.head())
print("Shape:", address_table.shape)

print("\n4) customer_table")
print(customer_table.head())
print("Shape:", customer_table.shape)

# Export the normalized tables to CSV
customer_csv_output = output_dir / "Customer.csv"
address_csv_output = output_dir / "Address.csv"

customer_table.to_csv(customer_csv_output, index=False, encoding="utf-8")
address_table.to_csv(address_csv_output, index=False, encoding="utf-8")

# Helper to format SQL values
def sql_literal(value):
    if pd.isna(value):
        return "NULL"
    return "'" + str(value).replace("'", "''") + "'"

# Export the normalized tables to SQL
customer_sql_output = output_dir / "Customer.sql"
address_sql_output = output_dir / "Address.sql"

with open(address_sql_output, "w", encoding="utf-8") as f:
    f.write(
        "CREATE TABLE Address (\n"
        "    ADDRESS_ID INT PRIMARY KEY,\n"
        "    ADDRESSLINE1 VARCHAR(255),\n"
        "    ADDRESSLINE2 VARCHAR(255),\n"
        "    CITY VARCHAR(100),\n"
        "    STATE VARCHAR(100),\n"
        "    POSTALCODE VARCHAR(20),\n"
        "    COUNTRY VARCHAR(100),\n"
        "    TERRITORY VARCHAR(50)\n"
        ");\n\n"
    )

    for _, row in address_table.iterrows():
        columns_str = ", ".join(address_table.columns)
        values_str = ", ".join(sql_literal(row[col]) for col in address_table.columns)
        f.write(f"INSERT INTO Address ({columns_str}) VALUES ({values_str});\n")

with open(customer_sql_output, "w", encoding="utf-8") as f:
    f.write(
        "CREATE TABLE Customer (\n"
        "    CUSTOMER_ID INT PRIMARY KEY,\n"
        "    CUSTOMERNAME VARCHAR(255),\n"
        "    PHONE VARCHAR(50),\n"
        "    ADDRESS_ID INT,\n"
        "    FOREIGN KEY (ADDRESS_ID) REFERENCES Address(ADDRESS_ID)\n"
        ");\n\n"
    )

    for _, row in customer_table.iterrows():
        columns_str = ", ".join(customer_table.columns)
        values_str = ", ".join(sql_literal(row[col]) for col in customer_table.columns)
        f.write(f"INSERT INTO Customer ({columns_str}) VALUES ({values_str});\n")

print("\nFiles created successfully:")
print("CSV:", customer_csv_output)
print("CSV:", address_csv_output)
print("SQL:", customer_sql_output)
print("SQL:", address_sql_output)

1) customer_base
               CUSTOMERNAME             PHONE                   ADDRESSLINE1  \
0         Land of Toys Inc.        2125557818        897 Long Airport Avenue   
1        Reims Collectables        26.47.1555             59 rue de l'Abbaye   
2           Lyon Souveniers  +33 1 46 62 7555  27 rue du Colonel Pierre Avia   
3         Toys4GrownUps.com        6265557265             78934 Hillside Dr.   
4  Corporate Gift Ideas Co.        6505551386                7734 Strong St.   

  ADDRESSLINE2           CITY STATE POSTALCODE COUNTRY TERRITORY  
0          NaN            NYC    NY      10022     USA       NaN  
1          NaN          Reims   NaN      51100  France      EMEA  
2          NaN          Paris   NaN      75508  France      EMEA  
3          NaN       Pasadena    CA      90003     USA       NaN  
4          NaN  San Francisco    CA        NaN     USA       NaN  
Shape: (92, 9)

2) customer_address_map
               CUSTOMERNAME             PHONE               

## Contact

In [6]:
# Preprocessing for Contact

# Define output directory
output_dir = Path(r'C:\Users\Alondra Soto\OneDrive\Documents\Development\Sample_Sales_Data\database\preprocessing_tables')
output_dir.mkdir(parents=True, exist_ok=True)

# Extract contact information
contact_columns = ['CONTACTLASTNAME', 'CONTACTFIRSTNAME', 'CUSTOMERNAME']
contact_base = df[contact_columns].copy()

# Create unique contact records with auto-incrementing ID
contact_unique = contact_base.drop_duplicates().reset_index(drop=True)
contact_unique.insert(0, 'CONTACT_ID', range(1, len(contact_unique) + 1))

# Create a mapping from customer to contact
contact_map = contact_base.merge(
    contact_unique,
    on=['CONTACTLASTNAME', 'CONTACTFIRSTNAME', 'CUSTOMERNAME'],
    how='left'
)

# Create final contact table (without CUSTOMERNAME for normalization)
contact_table = contact_unique[['CONTACT_ID', 'CONTACTLASTNAME', 'CONTACTFIRSTNAME']].copy()

# Show preprocessing steps
print("1) contact_base")
print(contact_base.head())
print("Shape:", contact_base.shape)

print("\n2) contact_unique")
print(contact_unique.head())
print("Shape:", contact_unique.shape)

print("\n3) contact_table")
print(contact_table.head())
print("Shape:", contact_table.shape)

# Export to CSV
contact_csv_output = output_dir / "Contact.csv"
contact_table.to_csv(contact_csv_output, index=False, encoding="utf-8")

# Helper to format SQL values
def sql_literal(value):
    if pd.isna(value):
        return "NULL"
    return "'" + str(value).replace("'", "''") + "'"

# Export to SQL
contact_sql_output = output_dir / "Contact.sql"

with open(contact_sql_output, "w", encoding="utf-8") as f:
    f.write(
        "CREATE TABLE Contact (\n"
        "    CONTACT_ID INT PRIMARY KEY,\n"
        "    CONTACTLASTNAME VARCHAR(100),\n"
        "    CONTACTFIRSTNAME VARCHAR(100),\n"
        "    CUSTOMER_ID INT,\n"
        "    FOREIGN KEY (CUSTOMER_ID) REFERENCES Customer(CUSTOMER_ID)\n"
        ");\n\n"
    )

    for _, row in contact_table.iterrows():
        columns_str = ", ".join(contact_table.columns)
        values_str = ", ".join(sql_literal(row[col]) for col in contact_table.columns)
        f.write(f"INSERT INTO Contact ({columns_str}) VALUES ({values_str});\n")

print("\nFiles created successfully:")
print("CSV:", contact_csv_output)
print("SQL:", contact_sql_output)


1) contact_base
  CONTACTLASTNAME CONTACTFIRSTNAME              CUSTOMERNAME
0              Yu             Kwai         Land of Toys Inc.
1         Henriot             Paul        Reims Collectables
2        Da Cunha           Daniel           Lyon Souveniers
3           Young            Julie         Toys4GrownUps.com
4           Brown            Julie  Corporate Gift Ideas Co.
Shape: (2823, 3)

2) contact_unique
   CONTACT_ID CONTACTLASTNAME CONTACTFIRSTNAME              CUSTOMERNAME
0           1              Yu             Kwai         Land of Toys Inc.
1           2         Henriot             Paul        Reims Collectables
2           3        Da Cunha           Daniel           Lyon Souveniers
3           4           Young            Julie         Toys4GrownUps.com
4           5           Brown            Julie  Corporate Gift Ideas Co.
Shape: (92, 4)

3) contact_table
   CONTACT_ID CONTACTLASTNAME CONTACTFIRSTNAME
0           1              Yu             Kwai
1           2    

## Domestic Customer

In [7]:
# Preprocessing for Domestic Customer (USA)


# Define output directory
output_dir = Path(r'C:\Users\Alondra Soto\OneDrive\Documents\Development\Sample_Sales_Data\database\preprocessing_tables')
output_dir.mkdir(parents=True, exist_ok=True)

# Filter for domestic customers (USA)
domestic_base = df[df['COUNTRY'] == 'USA'].copy()

# Extract relevant columns for domestic customers
domestic_columns = ['CUSTOMERNAME', 'PHONE', 'ADDRESSLINE1', 'ADDRESSLINE2', 'CITY', 'STATE', 'POSTALCODE', 'COUNTRY']
domestic_customer_base = domestic_base[domestic_columns].drop_duplicates().reset_index(drop=True)

# Add customer ID
domestic_customer_base.insert(0, 'CUSTOMER_ID', range(1, len(domestic_customer_base) + 1))

# Show preprocessing steps
print("1) domestic_base (filtered for USA)")
print(domestic_base.head())
print("Shape:", domestic_base.shape)
print("Unique customers:", domestic_customer_base.shape[0])

print("\n2) domestic_customer_base")
print(domestic_customer_base.head())
print("Shape:", domestic_customer_base.shape)

# Export to CSV
domestic_csv_output = output_dir / "DomesticCustomer.csv"
domestic_customer_base.to_csv(domestic_csv_output, index=False, encoding="utf-8")

# Helper to format SQL values
def sql_literal(value):
    if pd.isna(value):
        return "NULL"
    return "'" + str(value).replace("'", "''") + "'"

# Export to SQL
domestic_sql_output = output_dir / "DomesticCustomer.sql"

with open(domestic_sql_output, "w", encoding="utf-8") as f:
    f.write(
        "CREATE TABLE DomesticCustomer (\n"
        "    CUSTOMER_ID INT PRIMARY KEY,\n"
        "    FOREIGN KEY (CUSTOMER_ID) REFERENCES Customer(CUSTOMER_ID)\n"
        ");\n\n"
    )

    for _, row in domestic_customer_base.iterrows():
        columns_str = ", ".join(domestic_customer_base.columns)
        values_str = ", ".join(sql_literal(row[col]) for col in domestic_customer_base.columns)
        f.write(f"INSERT INTO DomesticCustomer ({columns_str}) VALUES ({values_str});\n")

print("\nFiles created successfully:")
print("CSV:", domestic_csv_output)
print("SQL:", domestic_sql_output)


1) domestic_base (filtered for USA)
   ORDERNUMBER  QUANTITYORDERED  PRICEEACH  ORDERLINENUMBER    SALES  \
0        10107               30      95.70                2  2871.00   
3        10145               45      83.26                6  3746.70   
4        10159               49     100.00               14  5205.27   
5        10168               36      96.66                1  3479.76   
8        10201               22      98.57                2  2168.54   

         ORDERDATE   STATUS  QTR_ID  MONTH_ID  YEAR_ID  ...  \
0   2/24/2003 0:00  Shipped       1         2     2003  ...   
3   8/25/2003 0:00  Shipped       3         8     2003  ...   
4  10/10/2003 0:00  Shipped       4        10     2003  ...   
5  10/28/2003 0:00  Shipped       4        10     2003  ...   
8   12/1/2003 0:00  Shipped       4        12     2003  ...   

                ADDRESSLINE1  ADDRESSLINE2           CITY STATE POSTALCODE  \
0    897 Long Airport Avenue           NaN            NYC    NY      10022

## International Customer

In [8]:
# Preprocessing for International Customer (Non-USA)

# Define output directory
output_dir = Path(r'C:\Users\Alondra Soto\OneDrive\Documents\Development\Sample_Sales_Data\database\preprocessing_tables')
output_dir.mkdir(parents=True, exist_ok=True)

# Filter for international customers (non-USA)
international_base = df[df['COUNTRY'] != 'USA'].copy()

# Extract relevant columns for international customers
international_columns = ['CUSTOMERNAME', 'PHONE', 'ADDRESSLINE1', 'ADDRESSLINE2', 'CITY', 'STATE', 'POSTALCODE', 'COUNTRY', 'TERRITORY']
international_customer_base = international_base[international_columns].drop_duplicates().reset_index(drop=True)

# Add customer ID
international_customer_base.insert(0, 'CUSTOMER_ID', range(1, len(international_customer_base) + 1))

# Show preprocessing steps
print("1) international_base (filtered for non-USA)")
print(international_base.head())
print("Shape:", international_base.shape)
print("Unique customers:", international_customer_base.shape[0])
print("\nCountries represented:")
print(international_base['COUNTRY'].value_counts())

print("\n2) international_customer_base")
print(international_customer_base.head())
print("Shape:", international_customer_base.shape)

# Export to CSV
international_csv_output = output_dir / "InternationalCustomer.csv"
international_customer_base.to_csv(international_csv_output, index=False, encoding="utf-8")

# Helper to format SQL values
def sql_literal(value):
    if pd.isna(value):
        return "NULL"
    return "'" + str(value).replace("'", "''") + "'"

# Export to SQL
international_sql_output = output_dir / "InternationalCustomer.sql"

with open(international_sql_output, "w", encoding="utf-8") as f:
    f.write(
        "CREATE TABLE InternationalCustomer (\n"
        "    CUSTOMER_ID INT PRIMARY KEY,\n"
        "    FOREIGN KEY (CUSTOMER_ID) REFERENCES Customer(CUSTOMER_ID)\n"
        ");\n\n"
    )

    for _, row in international_customer_base.iterrows():
        columns_str = ", ".join(international_customer_base.columns)
        values_str = ", ".join(sql_literal(row[col]) for col in international_customer_base.columns)
        f.write(f"INSERT INTO InternationalCustomer ({columns_str}) VALUES ({values_str});\n")

print("\nFiles created successfully:")
print("CSV:", international_csv_output)
print("SQL:", international_sql_output)


1) international_base (filtered for non-USA)
   ORDERNUMBER  QUANTITYORDERED  PRICEEACH  ORDERLINENUMBER    SALES  \
1        10121               34      81.35                5  2765.90   
2        10134               41      94.74                2  3884.34   
6        10180               29      86.13                9  2497.77   
7        10188               48     100.00                1  5512.32   
9        10211               41     100.00               14  4708.44   

         ORDERDATE   STATUS  QTR_ID  MONTH_ID  YEAR_ID  ...  \
1    5/7/2003 0:00  Shipped       2         5     2003  ...   
2    7/1/2003 0:00  Shipped       3         7     2003  ...   
6  11/11/2003 0:00  Shipped       4        11     2003  ...   
7  11/18/2003 0:00  Shipped       4        11     2003  ...   
9   1/15/2004 0:00  Shipped       1         1     2004  ...   

                    ADDRESSLINE1  ADDRESSLINE2    CITY STATE POSTALCODE  \
1             59 rue de l'Abbaye           NaN   Reims   NaN      51